# LLMDemo GPT-2 中文预训练（Colab/Kaggle）

使用步骤：
1. 菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**
2. 把 `llmdemo_pretrain_portable.tar.gz` 上传到你的 Google Drive 根目录
3. 依次运行下面的单元格

T4 上约 3 小时跑完 3900 步（≈2 个 epoch）。训练结束后运行最后一个单元格下载模型。

In [ ]:
# 1. 挂载 Google Drive 并解压迁移包
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/llmdemo && tar -xzf /content/drive/MyDrive/llmdemo_pretrain_portable.tar.gz -C /content/llmdemo
%cd /content/llmdemo
!ls -lh data/pretrain models/bpe_tokenizer

In [ ]:
# 2. 安装依赖（torch/numpy 已预装，通常几秒完成）
!pip -q install tokenizers
import torch; print('CUDA可用:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. 开始训练（GPU 显存 16GB，batch 32 x accum 4 = 每步 65,536 tokens，和 M1 上的设置等价）
# 如中途断线重连，把末尾注释的 --resume 加上即可从最近的 checkpoint 续训
!python3 pretrain.py --preset gpt2-demo --amp \
  --batch-size 32 --grad-accum 4 \
  --max-steps 3900 --lr 4e-4 --min-lr 4e-5 --warmup-steps 300 \
  --eval-interval 250 --ckpt-interval 250 --log-interval 50 \
  --out-dir models/gpt2_pretrain  # --resume

In [ ]:
# 4. 试一下生成效果
!python3 generate_gpt2.py --ckpt models/gpt2_pretrain/best.pt \
  --tokenizer-dir models/bpe_tokenizer --prompt "人工智能" --max-new-tokens 100

In [ ]:
# 5. 把训练好的模型存回 Google Drive（之后下载拷回原项目 models/gpt2_pretrain/）
!mkdir -p /content/drive/MyDrive/llmdemo_out
!cp models/gpt2_pretrain/ckpt.pt models/gpt2_pretrain/best.pt models/gpt2_pretrain/train.log /content/drive/MyDrive/llmdemo_out/ 2>/dev/null || cp models/gpt2_pretrain/* /content/drive/MyDrive/llmdemo_out/
!ls -lh /content/drive/MyDrive/llmdemo_out